# walk-assist v1: training and evaluation

In [ ]:
!pip install -q ultralytics==8.4.216

In [ ]:
import os, sys
IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.isdir('/kaggle/working')
WORK  = '/content' if IN_COLAB else '/kaggle/working' if IN_KAGGLE else os.path.abspath('work')
STORE = '/content/drive/MyDrive/Computer vision' if IN_COLAB else WORK
if STORE.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs(WORK, exist_ok=True); os.makedirs(STORE, exist_ok=True)

Mounted at /content/drive


In [ ]:
import ultralytics
print(ultralytics.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
8.4.216


In [ ]:
import os, glob, shutil
DS_V4, DS_V3 = f'{WORK}/ds_v4', f'{WORK}/ds_v3'
DS_V4_ZIP, DS_V3_ZIP = f'{STORE}/ds_v4.zip', f'{STORE}/ds_v3.zip'
if not os.path.isdir(f'{DS_V4}/train/images'):
    if os.path.exists(DS_V4_ZIP):
        shutil.unpack_archive(DS_V4_ZIP, WORK)
    else:
        if not os.path.isdir(f'{DS_V3}/train/images'):
            shutil.unpack_archive(DS_V3_ZIP, WORK)
        shutil.copytree(DS_V3, DS_V4)
        removed = []
        for sp in ['train', 'valid', 'test']:
            for f in glob.glob(f'{DS_V4}/{sp}/images/rpl*'):
                st = os.path.splitext(os.path.basename(f))[0]
                os.remove(f); removed.append((sp, os.path.basename(f)))
                lf = f'{DS_V4}/{sp}/labels/{st}.txt'
                if os.path.exists(lf): os.remove(lf)
        open(f'{DS_V4}/removed_rpl_watermark.txt', 'w').write('\n'.join(f'{s}\t{f}' for s, f in removed) + '\n')
        print('rpl removed:', len(removed), {s: sum(1 for x, _ in removed if x == s) for s in ['train', 'valid', 'test']})
        shutil.make_archive(DS_V4_ZIP[:-4], 'zip', WORK, 'ds_v4')
for sp in ['train', 'valid', 'test']:
    print(sp, len(os.listdir(f'{DS_V4}/{sp}/images')))

train 6247
valid 1163
test 881


In [ ]:
import glob, os, time, shutil, collections
import numpy as np, pandas as pd, torch, yaml
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou

# MODE='train': train + validation only.  MODE='test': evaluate the frozen model on test once.
MODE   = 'test'
RUNS   = [('yolo26s.pt', 640)]      
SEEDS  = [0, 1, 2]
EPOCHS = 50
BATCH  = 16
CONF   = 0.25
EXPECT = {'train': 6247, 'valid': 1163, 'test': 881}   
DVER   = 'v1120c_nowm'
AUTO_DISCONNECT = False    

SRC        = DS_V4
OUT        = f'{WORK}/runs'
BACKUP     = f'{STORE}/runs_backup'
SUBSET_DIR = f'{STORE}/audit_v3'
CSV        = f'{STORE}/val_compare_{DVER}.csv' if MODE == 'train' else f'{STORE}/test_compare_{DVER}.csv'
os.makedirs(OUT, exist_ok=True); os.makedirs(BACKUP, exist_ok=True)
print('data :', SRC, '\nmode :', MODE, '\ncsv  :', CSV)

data : /content/ds_v4 
mode : test 
csv  : /content/drive/MyDrive/Computer vision/test_compare_v1120c_nowm.csv


## 1. Data check


In [ ]:
names = yaml.safe_load(open(f'{SRC}/data.yaml'))['names']
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
DATA = f'{WORK}/data.yaml'
yaml.safe_dump(dict(path=SRC, train='train/images', val='valid/images', test='test/images',nc=len(names), names=names), open(DATA, 'w'), allow_unicode=True)

for sp in ['train', 'valid', 'test']:
    n_img = len([f for f in os.listdir(f'{SRC}/{sp}/images') if not f.startswith('.')])
    print(f'{sp:6} images {n_img}')
    assert n_img > 0 and (EXPECT is None or n_img == EXPECT[sp]), f'{sp}: check image count'
    if sp == 'valid': n_valid_img = n_img


cnt = collections.Counter(l.split()[0] for f in glob.glob(f'{SRC}/valid/labels/*.txt') for l in open(f) if l.strip())
n_box = sum(cnt.values())
print('valid boxes:', {names[int(k)]: v for k, v in sorted(cnt.items())}, 'total', n_box)
assert n_box > n_valid_img, 'box count <= image count -> check label files (newline bug?)'
print(open(DATA).read())

train  images 6247
valid  images 1163
test   images 881
valid boxes: {'crosswalk': 1636, 'green-crossing': 594, 'red-crossing': 456} total 2686
names:
- crosswalk
- green-crossing
- red-crossing
nc: 3
path: /content/ds_v4
test: test/images
train: train/images
val: valid/images



## 2. Evaluation functions
- Ultralytics metrics (overall and per-class P/R/AP50/AP50-95)
- Signal colour errors: for each GT signal, only the **single highest-confidence** signal prediction with IoU > 0.5 is judged (conf 0.25)
- **`red_as_green_%` = share of red signals predicted as green = a wrong "cross now" cue (safety metric)**

In [ ]:
GREEN, RED = names.index('green-crossing'), names.index('red-crossing')
CW = names.index('crosswalk')

def color_errors(weights, imgsz, split='valid'):
    m = YOLO(weights)
    st = {GREEN: [0, 0, 0], RED: [0, 0, 0]}          # [hit, wrong_colour, miss]
    wrong = []
    for r in m.predict(f'{SRC}/{split}/images', imgsz=imgsz, conf=CONF, stream=True, verbose=False, device=0):
        lf = f"{SRC}/{split}/labels/{os.path.splitext(os.path.basename(r.path))[0]}.txt"
        if not os.path.exists(lf) or os.path.getsize(lf) == 0: continue
        gt = np.loadtxt(lf, ndmin=2)
        h, w = r.orig_shape
        pb, pc, ps = r.boxes.xyxy.cpu(), r.boxes.cls.cpu().int(), r.boxes.conf.cpu()
        for c, x, y, bw, bh in gt[:, :5]:
            c = int(c)
            if c not in st: continue
            g = torch.tensor([[(x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h]])
            cand = (box_iou(g, pb)[0] > 0.5) & (pc != CW) if len(pb) else torch.zeros(0, dtype=torch.bool)
            if not cand.any(): st[c][2] += 1; continue
            best = torch.where(cand, ps, torch.tensor(-1.)).argmax()
            if pc[best] == c: st[c][0] += 1
            else: st[c][1] += 1; wrong.append((os.path.basename(r.path), names[c], float(ps[best])))
    return st, wrong

def model_size(weights, imgsz):
    try:
        from ultralytics.utils.torch_utils import get_flops
        m = YOLO(weights).model
        return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)
    except Exception as e:
        print('model size calculation failed (ignored):', e); return np.nan, np.nan

def evaluate(weights, tag, imgsz, split='valid', data=None):
    # split = 'valid' / 'test' / 'test_<subset>' (folder under SRC). data = yaml for subsets.
    assert MODE == 'test' or not split.startswith('test'), 'test must not be used before the model is frozen'
    v = YOLO(weights).val(data=data or DATA, split='val' if split == 'valid' else 'test', imgsz=imgsz, batch=16, device=0,project=OUT, name=f'{split}_{tag}', exist_ok=True, plots=True)
    row = dict(model=tag, split=split, imgsz=imgsz, mAP50=v.box.map50, mAP50_95=v.box.map,P=v.box.mp, R=v.box.mr, infer_ms=v.speed['inference'])
    for i, c in enumerate(v.box.ap_class_index):
        p, r, ap50, ap = v.box.class_result(i)
        n = names[int(c)]
        row.update({f'{n}_P': p, f'{n}_R': r, f'{n}_AP50': ap50, f'{n}_AP50_95': ap})
    st, wrong = color_errors(weights, imgsz, split)
    for c, n in [(RED, 'red'), (GREEN, 'green')]:
        ok, bad, miss = st[c]; t = max(1, sum(st[c]))
        row[f'{n}_wrongcolor_%'] = 100 * bad / t
        row[f'{n}_miss_%'] = 100 * miss / t
    row['red_as_green_%'] = row['red_wrongcolor_%']
    if wrong: print('wrong-colour signals:', wrong[:10])
    pd.DataFrame(wrong, columns=['image', 'gt', 'conf']).to_csv(f'{BACKUP}/{split}_{tag}_wrongcolor.csv', index=False)
    row['params_M'], row['GFLOPs'] = model_size(weights, imgsz)
    return row


## 3-A. Training + validation evaluation (MODE='train')


In [ ]:
def tag_of(w, imgsz, seed): return f'{os.path.splitext(w)[0]}_{imgsz}_s{seed}_{DVER}'

def backup(name):
    if os.path.isdir(f'{OUT}/{name}'):
        shutil.copytree(f'{OUT}/{name}', f'{BACKUP}/{name}', dirs_exist_ok=True)

def restore(name):
    if not os.path.isdir(f'{OUT}/{name}') and os.path.isdir(f'{BACKUP}/{name}'):
        shutil.copytree(f'{BACKUP}/{name}', f'{OUT}/{name}'); print(name, ': restored from backup')

def epochs_done(name):
    f = f'{OUT}/{name}/results.csv'
    return len(pd.read_csv(f)) if os.path.exists(f) else 0

def train_minutes(name):
    f = f'{OUT}/{name}/results.csv'
    if not os.path.exists(f): return np.nan
    df = pd.read_csv(f); df.columns = df.columns.str.strip()
    return df['time'].iloc[-1] / 60 if 'time' in df.columns else np.nan

def backup_every(name, k=5):
    def cb(trainer):
        if (trainer.epoch + 1) % k == 0: backup(name)
    return cb

if MODE == 'train':
    rows = pd.read_csv(CSV).to_dict('records') if os.path.exists(CSV) else []
    done = {r['model'] for r in rows}
    for seed in SEEDS:
        for w, imgsz in RUNS:
            tag = tag_of(w, imgsz, seed)
            if tag in done:
                print(tag, ': already in csv -> skipped'); continue
            restore(tag)
            best, last = f'{OUT}/{tag}/weights/best.pt', f'{OUT}/{tag}/weights/last.pt'
            ep = epochs_done(tag)
            if ep >= EPOCHS and os.path.exists(best):
                print(f'{tag}: training already finished ({ep} epochs) -> evaluate only')
            elif ep > 0 and os.path.exists(last):
                print(f'{tag}: resuming training from epoch {ep}')
                m = YOLO(last); m.add_callback('on_fit_epoch_end', backup_every(tag)); m.train(resume=True)
            else:
                m = YOLO(w); m.add_callback('on_fit_epoch_end', backup_every(tag))
                m.train(data=DATA, imgsz=imgsz, epochs=EPOCHS, batch=BATCH, device=0, workers=4,seed=seed, deterministic=False, project=OUT, name=tag, exist_ok=True)
            backup(tag)
            row = evaluate(best, tag, imgsz, 'valid')
            backup(f'valid_{tag}')
            row.update(seed=seed, train_min=train_minutes(tag))
            rows.append(row)
            pd.DataFrame(rows).to_csv(CSV, index=False)
            print(f'\n== {tag} done (val): mAP50 {row["mAP50"]:.3f}, mAP50-95 {row["mAP50_95"]:.3f}, '
                  f'green R {row.get("green-crossing_R", np.nan):.3f}, green miss {row["green_miss_%"]:.1f}% ==\n')
            torch.cuda.empty_cache()

## 3-B. Test evaluation (MODE='test', run once on the final model)

In [ ]:
if MODE == 'test':
    SUBSETS = ['full', 'no_seq', 'no_aug', 'clean', 'korea']
    sub_yaml = {}
    for name in SUBSETS:
        lst = open(f'{SUBSET_DIR}/subset_test_{name}.txt').read().split()
        d = f'{SRC}/test_{name}'
        os.makedirs(f'{d}/images', exist_ok=True); os.makedirs(f'{d}/labels', exist_ok=True)
        for f in lst:
            st_ = os.path.splitext(f)[0]
            for a, b in [(f'{SRC}/test/images/{f}', f'{d}/images/{f}'), (f'{SRC}/test/labels/{st_}.txt', f'{d}/labels/{st_}.txt')]:
                if os.path.exists(a) and not os.path.exists(b): os.symlink(a, b)
        y = f'{WORK}/data_test_{name}.yaml'
        yaml.safe_dump(dict(path=SRC, train='train/images', val='valid/images', test=f'test_{name}/images',nc=len(names), names=names), open(y, 'w'), allow_unicode=True)
        sub_yaml[name] = y
        print(f'test subset {name}: {len(os.listdir(d + "/images"))} images')
    rows = []
    for seed in SEEDS:
        for w, imgsz in RUNS:
            tag = tag_of(w, imgsz, seed)
            restore(tag)
            best = f'{OUT}/{tag}/weights/best.pt'
            if not os.path.exists(best):
                print(tag, ': no weights found (check BACKUP) -> skipped'); continue
            for name in SUBSETS:
                row = evaluate(best, f'{tag}_{name}', imgsz, f'test_{name}', data=sub_yaml[name])
                backup(f'test_{name}_{tag}_{name}')
                row.update(seed=seed, subset=name)
                rows.append(row)
                pd.DataFrame(rows).to_csv(CSV, index=False)
                print(f'{tag} [{name}] mAP50 {row["mAP50"]:.3f}  mAP50-95 {row["mAP50_95"]:.3f}  red_as_green {row["red_as_green_%"]:.2f}%')

test subset full: 881 images
test subset no_seq: 841 images
test subset no_aug: 823 images
test subset clean: 789 images
test subset korea: 492 images
yolo26s_640_s0_v1120c_nowm : restored from backup
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 120 layers, 9,466,341 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2160.4±590.3 MB/s, size: 128.7 KB)
val: Scanning /content/ds_v4/test_full/labels... 881 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 881/881 1.1Kit/s 0.8s
val: New cache created: /content/ds_v4/test_full/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 4.8it/s 11.7s
                   all        881       2149      0.872      0.829      0.864      0.515
             crosswalk        819       1338      0.865      0.778      0.875      0.626
        green-crossing        312        341      0

## 4. Comparison table (one row per run)

In [ ]:
df = pd.read_csv(CSV)
main = ['model', 'split', 'imgsz', 'mAP50', 'mAP50_95', 'P', 'R',
        'crosswalk_R', 'crosswalk_AP50_95', 'green-crossing_R', 'red-crossing_R',
        'green-crossing_AP50', 'red-crossing_AP50',
        'red_as_green_%', 'red_miss_%', 'green_wrongcolor_%', 'green_miss_%',
        'params_M', 'GFLOPs', 'train_min']
pd.set_option('display.width', 250)
print(df[[c for c in main if c in df.columns]].round(3).to_string(index=False))

                            model       split  imgsz  mAP50  mAP50_95     P     R  crosswalk_R  crosswalk_AP50_95  green-crossing_R  red-crossing_R  green-crossing_AP50  red-crossing_AP50  red_as_green_%  red_miss_%  green_wrongcolor_%  green_miss_%  params_M  GFLOPs
  yolo26s_640_s0_v1120c_nowm_full   test_full    640  0.864     0.515 0.872 0.829        0.778              0.626             0.853           0.856                0.845              0.873           0.426      12.340                 0.0        12.023      9.95  22.748
yolo26s_640_s0_v1120c_nowm_no_seq test_no_seq    640  0.868     0.515 0.878 0.831        0.778              0.626             0.857           0.859                0.855              0.875           0.440      12.088                 0.0        12.063      9.95  22.748
yolo26s_640_s0_v1120c_nowm_no_aug test_no_aug    640  0.870     0.520 0.878 0.831        0.777              0.627             0.857           0.860                0.853              0.882         

## 5. 3-seed mean ± std (report table)

In [ ]:
df = pd.read_csv(CSV)
cols = ['mAP50', 'mAP50_95', 'crosswalk_R', 'crosswalk_AP50_95', 'green-crossing_R', 'green-crossing_AP50','red-crossing_R', 'red-crossing_AP50', 'red_as_green_%', 'green_miss_%', 'red_miss_%']
cols = [c for c in cols if c in df.columns]
key = 'subset' if 'subset' in df.columns else 'imgsz'
g = df.groupby(key)[cols]
print('seeds:', sorted(df['seed'].unique()))
print((g.mean().round(3).astype(str) + ' ± ' + g.std().round(3).astype(str)).T.to_string())


seeds: [np.int64(0), np.int64(1), np.int64(2)]
subset                        clean            full           korea          no_aug          no_seq
mAP50                 0.877 ± 0.003    0.87 ± 0.006   0.862 ± 0.003   0.875 ± 0.004   0.874 ± 0.005
mAP50_95              0.523 ± 0.004    0.52 ± 0.004   0.492 ± 0.007   0.524 ± 0.004   0.519 ± 0.005
crosswalk_R           0.767 ± 0.009    0.77 ± 0.007   0.762 ± 0.011   0.771 ± 0.006   0.769 ± 0.008
crosswalk_AP50_95     0.624 ± 0.003   0.623 ± 0.003    0.61 ± 0.004   0.624 ± 0.003   0.623 ± 0.003
green-crossing_R      0.861 ± 0.002    0.85 ± 0.004    0.82 ± 0.003   0.855 ± 0.003   0.858 ± 0.003
green-crossing_AP50   0.862 ± 0.006    0.851 ± 0.01   0.839 ± 0.003    0.858 ± 0.01   0.862 ± 0.007
red-crossing_R        0.867 ± 0.004   0.861 ± 0.004   0.849 ± 0.006   0.866 ± 0.006   0.862 ± 0.003
red-crossing_AP50     0.891 ± 0.003   0.883 ± 0.009   0.872 ± 0.005   0.888 ± 0.006   0.884 ± 0.008
red_as_green_%         0.375 ± 0.13   0.355 ± 0.123  

## 6. Result figures: training curves + confusion matrix

In [ ]:
from IPython.display import Image, display
for w, imgsz in RUNS:
    tag = tag_of(w, imgsz, SEEDS[0])
    ev = f'{OUT}/test_full_{tag}_full' if MODE == 'test' else f'{OUT}/valid_{tag}'
    for f in [f'{OUT}/{tag}/results.png', f'{ev}/confusion_matrix_normalized.png']:
        if os.path.exists(f): print(f); display(Image(f, width=850))

In [ ]:
for name in os.listdir(OUT): backup(name)
print('Backup done:', BACKUP, '\ncsv:', CSV)
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()

Backup done: /content/drive/MyDrive/Computer vision/runs_backup 
csv: /content/drive/MyDrive/Computer vision/test_compare_v1120c_nowm.csv
